## Chains
**Chains are pipelines to connect I/O functions together like llms, tools etc.**

### Normal Chain

In [4]:
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

llm = ChatOllama(model="gemma4:31b-cloud",temperature=0)
parser = StrOutputParser()
template = PromptTemplate.from_template("""You are a helpgul assistant. Answer the quesrion {task}""")
chain = template | llm | parser

response = chain.invoke({"task":"What is the capital of france?"})
print(response)

The capital of France is **Paris**.


#### Generating graph

In [7]:
chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


### Sequential Chain

In [5]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


llm = ChatOllama(model="gemma4:31b-cloud")
parser = StrOutputParser()

template1 = PromptTemplate.from_template("Give me a short summary on topic: {topic}")
template2 = PromptTemplate.from_template("Based on the summary give me 5 keypoints on it \n{summary}")

seq_chain = template1 | llm | parser | template2 | llm | parser
seq_result = seq_chain.invoke({"topic":"Generative AI"})
print(seq_result)

Here are 5 key points based on the summary:

1.  **Core Purpose:** Unlike traditional AI that analyzes or classifies data, Generative AI focuses on creating entirely new, original content.
2.  **Technical Foundation:** It is powered by neural networks (such as Transformers and Diffusion models) and Large Language Models (LLMs) trained on massive datasets.
3.  **Versatile Outputs:** The technology can produce a wide range of media, including text, images, audio, and video.
4.  **Productivity Boost:** It offers significant benefits by automating routine tasks and speeding up creative and technical processes like brainstorming and coding.
5.  **Significant Risks:** The technology faces critical challenges, including "hallucinations" (false information), copyright issues, and the creation of deepfakes.


#### Generating graph of sequential chain

In [6]:
seq_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOllama |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

#### Trying out different models

In [8]:
llm1 = ChatOllama(model="gemma4:31b-cloud")
llm2 = ChatOllama(model="qwen2.5:0.5b")
parser = StrOutputParser()

template1 = PromptTemplate.from_template("Give me a short summary on topic: {topic}")
template2 = PromptTemplate.from_template("Based on the summary give me 5 keypoints on it \n{summary}")

seq_chain = template1 | llm1 | parser | template2 | llm2 | parser
seq_result = seq_chain.invoke({"topic":"Generative AI"})
print(seq_result)

Based on the provided summary, here are five key points about Generative AI:

1. **Focus Area**: Generative AI is a branch of artificial intelligence that is primarily concerned with creating new content rather than analyzing or classifying existing data.

2. **Technological Foundation**:
   - It relies heavily on **Foundation Models** and **Large Language Models (LLMs)** such as GPT-4, Stable Diffusion, among others.
   - These models are trained to recognize patterns in large datasets that can then be used to generate original content or creative outputs.

3. **Generated Outputs**:
   - It can produce diverse types of outputs including:
     - **Text:** Articles, code, poetry, and emails (e.g., ChatGPT, Claude).
     - **Images:** Digital art, photorealistic photos, and diagrams.
     - **Audio/Video:** Music composition, voice cloning, and synthetic video.

4. **Primary Uses**:
   - It is used for tasks such as automating repetitive tasks, brainstorming ideas, speeding up software d

#### So what I looked is 1st one same model loading and releasing twice took 10sec but 2nd one 2 different models took 16.4sec

## Parallel Chain
**To me it's really a important thing where many tasks occurs parallely**

In [11]:
from langchain_core.runnables import RunnableParallel

template1 = PromptTemplate.from_template("Give me a short summary on topic: {topic}")
template2 = PromptTemplate.from_template("Based on the summary give me 5 keypoints on it \n{topic}")

chain1 = template1 | llm | parser
chain2 = template2 | llm | parser

# Set up combiner
par_template = PromptTemplate.from_template("""
Combine following information into a comprehensive output:
summary: {summary}
key points: {key}
""")
parallel_chain = RunnableParallel({"summary":chain1,"key":chain2})


par_chain = par_template | llm | parser
final = parallel_chain | par_chain
result = final.invoke({"topic":"Computer science"})
print(result)

# Comprehensive Overview of Computer Science (CS)

## Definition and Purpose
**Computer Science (CS)** is the scientific study of computers and computational systems. While frequently equated with programming, it is a vast field encompassing the theory, design, development, and application of both software and hardware. 

The ultimate goal of computer science is to **automate problem-solving**. By translating complex real-world challenges into a language machines can process, computer scientists create tools that drive scientific discovery, increase global efficiency, and power the modern digital economy.

---

## The Five Pillars of Computer Science
The field is built upon five foundational pillars that bridge the gap between theoretical mathematics and practical application:

1.  **Algorithm Design & Complexity:** The creation of step-by-step procedures to solve problems, with a heavy focus on efficiency (analyzing time and memory usage via Big O notation).
2.  **Software Engineering

#### Checking it's flow chart

In [12]:
final.get_graph().print_ascii()

           +----------------------------+            
           | Parallel<summary,key>Input |            
           +----------------------------+            
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                    +------------+   
  | ChatOllama |                    | ChatOllama |   
  +------------+                    +------------+   
          *                               *          
          *                               *          
          *                               *          
+-----------------+         

## Conditional chain
**Works based on feedback**

### Get conditions

In [17]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser
from typing import Literal, Optional
from langchain_core.runnables import RunnableBranch

class Review(BaseModel):
    sentiment: Literal["positive","negative","neutral"] = Field(description="The sentiment of the movie review")

str_parser = StrOutputParser()
pydantice_parser = PydanticOutputParser(pydantic_object=Review)

con_template1 = PromptTemplate.from_template("""
Give me sentiment of the movie review {review} \n
format_instruction = {format_instruction}""",
partial_variables={"format_instruction":pydantice_parser.get_format_instructions})
review_chain = con_template1 | llm |pydantice_parser
#print(review_chain.invoke({"review":"The movie was awesome"}))

### Run on behalf of sentiment

In [18]:
pos_template = PromptTemplate.from_template("The review is positive so write a short appreciation \n {review}")
neg_template = PromptTemplate.from_template("The review is negative so write a short critique \n {review}")
neu_template = PromptTemplate.from_template("The review is neutral so write a short balance remark \n {review}")
default_temp = PromptTemplate.from_template("I couldn't understand so provide a neutral response \n {review}")

branch = RunnableBranch(
    (lambda x:x.sentiment=="positive", pos_template | llm | str_parser),
    (lambda x:x.sentiment=="negative", neg_template | llm | str_parser),
    (lambda x:x.sentiment=="neutral", neu_template | llm | str_parser),
    default_temp | llm | str_parser
)

final_con_chain = review_chain | branch


prompt = "The movie was good, nice vfx, camera movements but there was problems in the scripting I guess overall nice experience"
response = final_con_chain.invoke({"review":prompt})
print(response)

Depending on who the message is for, here are a few options:

**Option 1: Professional & Polished (Best for a business/client)**
"Thank you so much for your kind words! We're thrilled to hear you had a great experience and appreciate you taking the time to share your feedback."

**Option 2: Short & Sweet (Best for social media/quick replies)**
"Thanks for the lovely review! We're so glad you enjoyed it. 😊"

**Option 3: Enthusiastic (Best for a loyal customer)**
"Wow, thank you! We are so happy you're pleased with the results. We can't wait to serve you again soon!"


In [19]:
# trying tobe negative

prompt = "The movie was bad, literally one of my worst experience"
response = final_con_chain.invoke({"review":prompt})
print(response)

Since you didn't provide the specific text of the review, I have provided three options depending on what you are critiquing. Choose the one that fits best:

**Option 1: Professional & Polite (Best for a business response)**
"The feedback provided is disappointing and highlights significant shortcomings in the quality of service. The experience failed to meet basic expectations, leaving much room for improvement."

**Option 2: Sharp & Direct (Best for a critique of a product/movie/book)**
"The experience was thoroughly underwhelming. From the lack of attention to detail to the poor execution, it is clear that the end result lacks the quality and care necessary to be successful."

**Option 3: Short & Punchy (Best for a quick summary)**
"A disappointing failure in execution. The flaws outweigh the benefits, resulting in a subpar experience that cannot be recommended."

***

**If you paste the actual review, I can write a much more specific critique for you!**


### Lets get it's flow chart

In [21]:
final_con_chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOllama |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +--------------+     
